In [1]:
import os

# Caminho absoluto da pasta onde o script está sendo executado
diretorio_atual = os.getcwd()

print("Diretório atual:", diretorio_atual)

Diretório atual: /workspaces/IC-RNA-2025/fuction_ensemble_5/media_ponderada


In [1]:
import pandas as pd
import numpy as np
import ast
from sklearn.metrics import mean_squared_error

# ================================
# 1) CONFIGURAÇÕES
# ================================
arquivo_pesos = "/workspaces/IC-RNA-2025/fuction_ensemble_5/media_ponderada/todos_resultados_50_lm3.xlsx"
diretorio_redes = "/workspaces/IC-RNA-2025/fuction_ensemble_5/media_ponderada/1000_model"
limiar_peso = 0.1

df_pesos = pd.read_excel(arquivo_pesos).head(50)

resultados = []

for idx, row in df_pesos.iterrows():
    # Coluna A = nomes das redes, Coluna E = pesos
    nomes_str = row.iloc[0]
    pesos_str = row.iloc[4]

    # 🔧 Converte string "[a b c]" → lista de strings → lista de floats
    nomes_redes = nomes_str.strip("[]").replace("'", "").split(",")
    nomes_redes = [nome.strip() for nome in nomes_redes]

    pesos = np.array(pesos_str.strip("[]").split(), dtype=float)


    # ===============================
    # 2) CARREGA Z E Z_pred
    # ===============================
    dfs = [pd.read_excel(f"{diretorio_redes}/{nome}") for nome in nomes_redes]
    Z = dfs[0]["Z"].values.reshape(-1, 1)
    Z_preds = np.hstack([df["Z_pred"].values.reshape(-1, 1) for df in dfs])

    # ===============================
    # 3) MSE COM TODAS AS REDES
    # ===============================
    Z_pred_ponderada = Z_preds @ pesos
    mse_sup = mean_squared_error(Z, Z_pred_ponderada)

    # ===============================
    # 4) REMOVE REDES COM PESO < 0.1
    # ===============================
    mask = pesos >= 0.1
    nomes_filtrados = [nome for nome, keep in zip(nomes_redes, mask) if keep]
    pesos_filtrados = pesos[mask]
    pesos_filtrados = pesos_filtrados / np.sum(pesos_filtrados)

    Z_preds_filtrados = Z_preds[:, mask]
    yhat_filtrado = Z_preds_filtrados @ pesos_filtrados
    mse_filtrado = mean_squared_error(Z, yhat_filtrado)

    resultados.append({
        "linha": idx + 1,
        "MSE_sup": mse_sup,
        "MSE_sup_novo": mse_filtrado,
        "num_redes_total": len(nomes_redes),
        "num_redes_filtradas": len(nomes_filtrados),
        "nome_redes_filtradas": nomes_filtrados
    })


# ===============================
# 5) RESULTADOS EM DATAFRAME
# ===============================
df_resultados = pd.DataFrame(resultados)
print(df_resultados)

# Salva se quiser
df_resultados.to_excel("comparacao_mse_50_ensembles.xlsx", index=False)

    linha   MSE_sup  MSE_sup_novo  num_redes_total  num_redes_filtradas  \
0       1  0.033362      0.033405               10                    4   
1       2  0.033573      0.033557               10                    6   
2       3  0.033579      0.033571               10                    4   
3       4  0.033635      0.033678               10                    4   
4       5  0.033676      0.033676               10                    4   
5       6  0.033800      0.034052               10                    5   
6       7  0.033819      0.033854               10                    6   
7       8  0.033856      0.033856               10                    6   
8       9  0.033858      0.034233               10                    5   
9      10  0.033902      0.033890               10                    5   
10     11  0.033909      0.033885               10                    5   
11     12  0.033975      0.034257               10                    4   
12     13  0.033991      

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import os

diretorio_redes_1000 = "/workspaces/IC-RNA-2025/fuction_ensemble_5/media_ponderada/1000_model"
diretorio_redes_25 = "/workspaces/IC-RNA-2025/fuction_ensemble_5/media_ponderada/25_model"
arquivo_pesos = "/workspaces/IC-RNA-2025/fuction_ensemble_5/media_ponderada/todos_resultados_50_lm3_filtrada_covar.xlsx"
limiar_peso = 0.1

df_pesos = pd.read_excel(arquivo_pesos)

resultados = []

# ===============================
# FUNÇÃO AUXILIAR
# ===============================
def extrai_id(nome):
    return nome.split("model_")[1]


# ===============================
# FUNÇÃO DE AVALIAÇÃO COMPLETA (FILTRADO)
# ===============================
def avalia_filtrado(nomes_filtrados, pesos_filtrados):

    ids = [extrai_id(nome) for nome in nomes_filtrados]

    nomes_1000 = [f"1000_model_{id}" for id in ids]
    nomes_25 = [f"25_model_{id}" for id in ids]

    # ---------- 1000 ----------
    df_list1 = [pd.read_excel(os.path.join(diretorio_redes_1000, nome)) for nome in nomes_1000]
    z_preds_1000 = np.hstack([df['Z_pred'].values.reshape(-1, 1) for df in df_list1])
    Z_1000 = df_list1[0]['Z'].values.reshape(-1, 1)

    # ---------- 25 ----------
    df_list2 = [pd.read_excel(os.path.join(diretorio_redes_25, nome)) for nome in nomes_25]
    z_preds_25 = np.hstack([df['Z_pred'].values.reshape(-1, 1) for df in df_list2])
    Z_25 = df_list2[0]['Z'].values.reshape(-1, 1)

    yhat_1000 = z_preds_1000 @ pesos_filtrados
    yhat_25 = z_preds_25 @ pesos_filtrados

    mse_1000 = np.mean((Z_1000.flatten() - yhat_1000.flatten()) ** 2)
    r2_1000 = r2_score(Z_1000, yhat_1000)

    mse_25 = np.mean((Z_25.flatten() - yhat_25.flatten()) ** 2)
    r2_25 = r2_score(Z_25, yhat_25)

    M = z_preds_25.shape[1]

    var = np.mean([(z_preds_25[:, i] - yhat_25.flatten()) ** 2 for i in range(M)])
    bias = np.mean(yhat_25.flatten() - Z_25.flatten())

    cov_sum = 0
    for i in range(M):
        for j in range(M):
            if i != j:
                cov_sum += np.mean(
                    (z_preds_25[:, i] - z_preds_25[:, i].mean()) *
                    (z_preds_25[:, j] - z_preds_25[:, j].mean())
                )

    covar = cov_sum / (M * (M - 1)) if M > 1 else 0

    return mse_1000, r2_1000, r2_25, mse_25, var, bias, covar


# ===============================
# LOOP PRINCIPAL
# ===============================
for idx, row in df_pesos.iterrows():

    nomes_str = row.iloc[0]
    pesos_str = row.iloc[4]

    nomes_redes = eval(nomes_str)

    pesos = np.fromstring(
        pesos_str.strip("[]"),
        sep=" "
    )

    # ===============================
    # 🔵 MSE 1000 ORIGINAL (SEM FILTRO)
    # ===============================
    ids_orig = [extrai_id(nome) for nome in nomes_redes]
    nomes_1000_orig = [f"1000_model_{id}" for id in ids_orig]

    df_list_orig = [
        pd.read_excel(os.path.join(diretorio_redes_1000, nome))
        for nome in nomes_1000_orig
    ]

    z_preds_1000_orig = np.hstack([
        df['Z_pred'].values.reshape(-1, 1)
        for df in df_list_orig
    ])

    Z_1000_orig = df_list_orig[0]['Z'].values.reshape(-1, 1)

    yhat_1000_orig = z_preds_1000_orig @ pesos
    mse_1000_original = np.mean(
        (Z_1000_orig.flatten() - yhat_1000_orig.flatten()) ** 2
    )

    # ===============================
    # FILTRAGEM
    # ===============================
    mask = pesos >= limiar_peso
    nomes_filtrados = [nome for nome, keep in zip(nomes_redes, mask) if keep]
    pesos_filtrados = pesos[mask]

    if len(pesos_filtrados) == 0:
        continue

    pesos_filtrados = pesos_filtrados / np.sum(pesos_filtrados)

    # ===============================
    # AVALIAÇÃO FILTRADA
    # ===============================
    mse_1000_f, r2_1000_f, r2_25_f, mse_25_f, var, bias, covar = \
        avalia_filtrado(nomes_filtrados, pesos_filtrados)

    resultados.append({
        "linha": idx + 1,
        "mse_1000_original": mse_1000_original,   
        "mse_1000_f": mse_1000_f,
        "r2_1000_f": r2_1000_f,
        "mse_25_f": mse_25_f,
        "r2_25_f": r2_25_f,
        "var_f": var,
        "bias_f": bias,
        "covar_f": covar,
        "num_redes_total": len(nomes_redes),
        "num_redes_filtradas": len(nomes_filtrados),
        "nomes_redes_filtradas": nomes_filtrados,
        "pesos_filtrados": pesos_filtrados
    })

# ===============================
# SALVA RESULTADOS
# ===============================
df_resultados = pd.DataFrame(resultados)
print(df_resultados)

df_resultados.to_excel("metricas_ensemble_filtrado.xlsx", index=False)

    linha  mse_1000_original  mse_1000_f  r2_1000_f  mse_25_f   r2_25_f  \
0       1           0.035954    0.035692   0.964578  0.000748  0.999258   
1       2           0.036095    0.035711   0.964560  0.000435  0.999569   
2       3           0.036208    0.036322   0.963953  0.000429  0.999575   
3       4           0.036473    0.036714   0.963564  0.000500  0.999504   
4       5           0.036595    0.040920   0.959390  0.000133  0.999868   
5       6           0.036620    0.037977   0.962311  0.000250  0.999752   
6       7           0.036672    0.036928   0.963352  0.000601  0.999404   
7       8           0.037647    0.038736   0.961558  0.001276  0.998734   
8       9           0.041736    0.041736   0.958581  0.001956  0.998059   
9      10           0.043452    0.045508   0.954837  0.002975  0.997048   
10     11           0.045387    0.045508   0.954837  0.002975  0.997048   
11     12           0.045399    0.045508   0.954837  0.002975  0.997048   
12     13           0.045